In [1]:
#!pip install ultralytics opencv-python-headless pandas tqdm tabulate -q

In [2]:
#!pip install -i https://mirrors.aliyun.com/pypi/simple/ tabulate -q

In [3]:
import os
import cv2
import numpy as np
import torch
import re
import warnings
from pathlib import Path
from ultralytics import YOLO
from tqdm.notebook import tqdm
import pandas as pd
import sys
sys.path.append("./")
from ocr_predictor import OCRPredictor

warnings.filterwarnings('ignore')

# НАСТРОЙКИ

YOLO_MODEL_PATH = "./yolo11_best.pt"  # путь модели детекции
OCR_MODEL_PATH = "./best_ocr_crnn.pth" # путь модели ocr

CONF_THRESHOLDS = {
    "high": 0.85,    # sure
    "medium": 0.60   # maybe
}

# ВВОД ДАННЫХ

IMG_DIR = input("📁 Введите путь к папке с изображениями: ").strip()
PREFIX = input("🏷 Введите префикс для имён файлов (или нажмите Enter): ").strip()

if not IMG_DIR:
    raise ValueError("Путь к папке не указан.")

# ЗАГРУЗКА МОДЕЛЕЙ

print("⏳ Загрузка YOLO11...")
yolo = YOLO(YOLO_MODEL_PATH)

print("⏳ Загрузка CRNN...")
ocr_model = OCRPredictor(OCR_MODEL_PATH)
print("✅ Модели загружены.")

# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ

# Определяем категорию уверенности
def get_confidence_category(conf):
    if conf >= CONF_THRESHOLDS["high"]:
        return "sure"
    elif conf >= CONF_THRESHOLDS["medium"]:
        return "maybe"
    else:
        return "not sure"

# Удаляем недопустимые символы для имён файлов
def clean_filename(text, prefix):
    text = re.sub(r'[<>:"/\\|?*\x00-\x1F]', '_', text).strip()
    if not text:
        text = "unknown"
    return f"{prefix}_{text}" if prefix else text

# Генерируем уникальное имя файла с суффиксами _1, _2 и т.д.
def get_unique_path(directory, stem, ext):
    candidate = directory / f"{stem}{ext}"
    counter = 1
    while candidate.exists():
        candidate = directory / f"{stem}_{counter}{ext}"
        counter += 1
    return candidate

# ОСНОВНОЙ ПАЙПЛАЙН

img_dir = Path(IMG_DIR)
if not img_dir.is_dir():
    raise FileNotFoundError(f"❌ Папка не найдена: {IMG_DIR}")

# Фильтрация только изображений
extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp')
img_paths = [p for p in img_dir.iterdir() if p.suffix.lower() in extensions]

results = []

print(f"📊 Найдено изображений: {len(img_paths)}. Запуск обработки...")
for img_path in tqdm(img_paths, desc="Обработка"):
    img = cv2.imread(str(img_path))
    if img is None:
        continue

    # 1. YOLO11 Inference
    yolo_res = yolo(img, verbose=False, device='cpu')
    boxes = yolo_res[0].boxes

    if len(boxes) == 0:
        results.append({
            "original_file": img_path.name, "status": "no_detection",
            "ocr_text": "", "ocr_confidence": 0.0, "category": "no detection"
        })
        continue

    # Берём только один бокс с максимальной уверенностью
    best_idx = int(boxes.conf.argmax())
    xyxy = boxes.xyxy[best_idx].cpu().numpy().astype(int)
    yolo_conf = boxes.conf[best_idx].item()

    x1, y1, x2, y2 = xyxy
    h, w = img.shape[:2]
    # Обрезаем границы, чтобы не выйти за пределы изображения
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)

    crop = img[y1:y2, x1:x2]
    if crop.size == 0:
        continue

    # 2. CRNN Inference
    text, ocr_conf = ocr_model.predict(crop)
    category = get_confidence_category(ocr_conf)

    # 3. Переименование файла
    ext = img_path.suffix
    new_stem = clean_filename(text, PREFIX)
    new_path = get_unique_path(img_dir, new_stem, ext)

    try:
        img_path.rename(new_path)
        final_name = new_path.name
    except Exception as e:
        final_name = f"ERROR_{img_path.name}"
        print(f"⚠️ Ошибка переименования {img_path.name}: {e}")

    results.append({
        "original_file": img_path.name,
        "new_file": final_name,
        "yolo_conf": round(yolo_conf, 4),
        "ocr_text": text,
        "ocr_confidence": round(ocr_conf, 4),
        "category": category
    })

# ВЫВОД РЕЗУЛЬТАТОВ
df_results = pd.DataFrame(results)

# Функция для подсветки строк по категории
def highlight_row(row):
    """Возвращает список стилей для каждой колонки строки"""
    if row['category'] == 'sure':
        color = 'background-color: #d4edda'  # зелёный
    elif row['category'] == 'maybe':
        color = 'background-color: #fff3cd'  # жёлтый
    else:
        color = 'background-color: #f8d7da'  # красный
    # Возвращаем список стилей длиной ровно в количество колонок
    return [color] * len(row)

# Отображаем таблицу с форматированием
display(df_results.style.apply(highlight_row, axis=1))

# Быстрая статистика по категориям
print("\n📈 Статистика по уверенности:")
print(df_results['category'].value_counts().to_markdown())

# Сохраняем отчёт в CSV
df_results.to_csv(img_dir / "report.csv", index=False, sep=';')

print(f"\n✅ Обработка завершена! Успешно переименовано: {len([r for r in results if 'ERROR' not in r.get('new_file', '')])} файлов.")

📁 Введите путь к папке с изображениями:  C:\dev\sefer_project\project\TEST
🏷 Введите префикс для имён файлов (или нажмите Enter):  


⏳ Загрузка YOLO11...
⏳ Загрузка CRNN...
✅ Модели загружены.
📊 Найдено изображений: 192. Запуск обработки...


Обработка:   0%|          | 0/192 [00:00<?, ?it/s]

,original_file,status,ocr_text,ocr_confidence,category,new_file,yolo_conf
0,MDZ088 _4.JPG,no_detection,,0.000000,no detection,nan,nan
1,MDZ088 _6.JPG,no_detection,,0.000000,no detection,nan,nan
2,MDZ088 _7.JPG,no_detection,,0.000000,no detection,nan,nan
3,MDZ089 _5.JPG,no_detection,,0.000000,no detection,nan,nan
4,MDZ089 _6.JPG,no_detection,,0.000000,no detection,nan,nan
5,MDZ091 _4.JPG,no_detection,,0.000000,no detection,nan,nan
6,MDZ091 _5.JPG,no_detection,,0.000000,no detection,nan,nan
7,MDZ095 _3.JPG,no_detection,,0.000000,no detection,nan,nan
8,MDZ097_4.JPG,no_detection,,0.000000,no detection,nan,nan
9,MDZ099_5.JPG,no_detection,,0.000000,no detection,nan,nan



📈 Статистика по уверенности:
| category     |   count |
|:-------------|--------:|
| sure         |     165 |
| no detection |      17 |
| maybe        |       7 |
| not sure     |       3 |

✅ Обработка завершена! Успешно переименовано: 192 файлов.
